# 05 · Filtering, Sorting, and Modifying Data

**Goal:** go beyond basic selection — apply functions across columns, sort data, bin
continuous values, and reshape column values using `map`/`apply`/`replace`.

### Setup

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana", "Evan"],
    "age": [25, 32, 18, 47, 29],
    "department": ["Sales", "Engineering", "Sales", "Marketing", "Engineering"],
    "salary": [55000, 85000, 48000, 62000, 91000]
})
print(df)

      name  age   department  salary
0    Alice   25        Sales   55000
1      Bob   32  Engineering   85000
2  Charlie   18        Sales   48000
3    Diana   47    Marketing   62000
4     Evan   29  Engineering   91000


### Sorting

`.sort_values()` sorts by one or more columns; `.sort_index()` sorts by the row labels.

In [2]:
print(df.sort_values("age"))                      # ascending by default
print()
print(df.sort_values("age", ascending=False))       # descending
print()
print(df.sort_values(["department", "age"]))         # sort by MULTIPLE columns (like SQL ORDER BY)

      name  age   department  salary
2  Charlie   18        Sales   48000
0    Alice   25        Sales   55000
4     Evan   29  Engineering   91000
1      Bob   32  Engineering   85000
3    Diana   47    Marketing   62000

      name  age   department  salary
3    Diana   47    Marketing   62000
1      Bob   32  Engineering   85000
4     Evan   29  Engineering   91000
0    Alice   25        Sales   55000
2  Charlie   18        Sales   48000

      name  age   department  salary
4     Evan   29  Engineering   91000
1      Bob   32  Engineering   85000
3    Diana   47    Marketing   62000
2  Charlie   18        Sales   48000
0    Alice   25        Sales   55000


In [3]:
df_shuffled = df.sample(frac=1, random_state=1)   # shuffle the rows to demonstrate sort_index
print(df_shuffled)
print()
print(df_shuffled.sort_index())                     # restores the original row order

      name  age   department  salary
2  Charlie   18        Sales   48000
1      Bob   32  Engineering   85000
4     Evan   29  Engineering   91000
0    Alice   25        Sales   55000
3    Diana   47    Marketing   62000

      name  age   department  salary
0    Alice   25        Sales   55000
1      Bob   32  Engineering   85000
2  Charlie   18        Sales   48000
3    Diana   47    Marketing   62000
4     Evan   29  Engineering   91000


### Modifying values in place: `.map()`, `.apply()`, `.replace()`

- **`.map()`** — works on a `Series`; applies a function or dict/lookup to every element.
- **`.apply()`** — works on a `Series` OR a `DataFrame` (row-wise or column-wise with `axis`).
- **`.replace()`** — swaps specific values for other values.

In [4]:
# .map() with a dictionary -- great for recoding categories
dept_codes = {"Sales": "S", "Engineering": "E", "Marketing": "M"}
df["dept_code"] = df["department"].map(dept_codes)
print(df)

      name  age   department  salary dept_code
0    Alice   25        Sales   55000         S
1      Bob   32  Engineering   85000         E
2  Charlie   18        Sales   48000         S
3    Diana   47    Marketing   62000         M
4     Evan   29  Engineering   91000         E


In [5]:
# .map() with a function
df["age_in_5_years"] = df["age"].map(lambda x: x + 5)
print(df[["name", "age", "age_in_5_years"]])

      name  age  age_in_5_years
0    Alice   25              30
1      Bob   32              37
2  Charlie   18              23
3    Diana   47              52
4     Evan   29              34


In [6]:
# .apply() on a Series -- same idea as .map() for simple functions
df["salary_k"] = df["salary"].apply(lambda x: x / 1000)
print(df[["name", "salary", "salary_k"]])

# .apply() on a whole DataFrame with axis=1 -- runs a function on each ROW
df["summary"] = df.apply(lambda row: f"{row['name']} ({row['department']})", axis=1)
print(df[["name", "department", "summary"]])

      name  salary  salary_k
0    Alice   55000      55.0
1      Bob   85000      85.0
2  Charlie   48000      48.0
3    Diana   62000      62.0
4     Evan   91000      91.0
      name   department             summary
0    Alice        Sales       Alice (Sales)
1      Bob  Engineering   Bob (Engineering)
2  Charlie        Sales     Charlie (Sales)
3    Diana    Marketing   Diana (Marketing)
4     Evan  Engineering  Evan (Engineering)


In [7]:
# .replace() -- swap specific values
df_copy = df.copy()
df_copy["department"] = df_copy["department"].replace({"Sales": "Retail"})
print(df_copy["department"])

0         Retail
1    Engineering
2         Retail
3      Marketing
4    Engineering
Name: department, dtype: str


### Binning continuous data with `pd.cut`

Turning a numeric column into categories (e.g. age groups) is an extremely common data
preparation step.

In [8]:
bins = [0, 25, 40, 100]
labels = ["Young", "Middle-aged", "Senior"]

df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)
print(df[["name", "age", "age_group"]])

      name  age    age_group
0    Alice   25        Young
1      Bob   32  Middle-aged
2  Charlie   18        Young
3    Diana   47       Senior
4     Evan   29  Middle-aged


### Conditional column creation: `np.where` and `np.select`

`np.where` works exactly like it did in NumPy notebook 06 — pandas Series are built on
NumPy arrays under the hood.

In [9]:
df["salary_tier"] = np.where(df["salary"] >= 70000, "High", "Standard")
print(df[["name", "salary", "salary_tier"]])

# np.select for MORE than two conditions
conditions = [
    df["salary"] < 55000,
    (df["salary"] >= 55000) & (df["salary"] < 80000),
    df["salary"] >= 80000
]
choices = ["Low", "Mid", "High"]
df["salary_band"] = np.select(conditions, choices, default="Unknown")
print(df[["name", "salary", "salary_band"]])

      name  salary salary_tier
0    Alice   55000    Standard
1      Bob   85000        High
2  Charlie   48000    Standard
3    Diana   62000    Standard
4     Evan   91000        High
      name  salary salary_band
0    Alice   55000         Mid
1      Bob   85000        High
2  Charlie   48000         Low
3    Diana   62000         Mid
4     Evan   91000        High


### Filtering with string methods (the `.str` accessor)

Any Series of strings gets a `.str` accessor exposing vectorized string operations —
no manual loops needed.

In [10]:
df["name_upper"] = df["name"].str.upper()
df["name_length"] = df["name"].str.len()
df["starts_with_a"] = df["name"].str.startswith("A")

print(df[["name", "name_upper", "name_length", "starts_with_a"]])

# Filtering using a string condition
print(df[df["name"].str.contains("a", case=False)])   # names containing "a" (case-insensitive)

      name name_upper  name_length  starts_with_a
0    Alice      ALICE            5           True
1      Bob        BOB            3          False
2  Charlie    CHARLIE            7          False
3    Diana      DIANA            5          False
4     Evan       EVAN            4          False
      name  age   department  salary dept_code  age_in_5_years  salary_k  \
0    Alice   25        Sales   55000         S              30      55.0   
2  Charlie   18        Sales   48000         S              23      48.0   
3    Diana   47    Marketing   62000         M              52      62.0   
4     Evan   29  Engineering   91000         E              34      91.0   

              summary    age_group salary_tier salary_band name_upper  \
0       Alice (Sales)        Young    Standard         Mid      ALICE   
2     Charlie (Sales)        Young    Standard         Low    CHARLIE   
3   Diana (Marketing)       Senior    Standard         Mid      DIANA   
4  Evan (Engineering)  Midd

### Renaming, reordering, and dropping columns

In [11]:
# Reorder columns explicitly
df_reordered = df[["name", "department", "age", "salary"]]
print(df_reordered)

# Drop several columns we added along the way, back to a clean state
df_clean = df.drop(columns=["dept_code", "age_in_5_years", "salary_k", "summary",
                              "age_group", "salary_tier", "salary_band",
                              "name_upper", "name_length", "starts_with_a"])
print(df_clean)

      name   department  age  salary
0    Alice        Sales   25   55000
1      Bob  Engineering   32   85000
2  Charlie        Sales   18   48000
3    Diana    Marketing   47   62000
4     Evan  Engineering   29   91000
      name  age   department  salary
0    Alice   25        Sales   55000
1      Bob   32  Engineering   85000
2  Charlie   18        Sales   48000
3    Diana   47    Marketing   62000
4     Evan   29  Engineering   91000


### 🧠 Quick check

1. What's the difference between `.map()` and `.apply()` when used on a Series?
2. When would you use `np.select` instead of `np.where`?
3. What does the `.str` accessor let you do?

<details>
<summary>Answers</summary>

1. For simple element-wise transformations they behave similarly, but `.map()` is Series-only
   and commonly used with a dict lookup; `.apply()` works on both Series and DataFrames (with
   `axis` control for row-wise/column-wise application on a DataFrame).
2. `np.where` only handles a single True/False condition (two outcomes); `np.select` handles
   multiple conditions with multiple corresponding outcomes.
3. It exposes vectorized string operations (`.upper()`, `.len()`, `.contains()`, `.startswith()`,
   etc.) that apply to every element in a string Series at once.
</details>

### ✍️ Practice

1. Sort the sample DataFrame by `salary` descending, then by `department` ascending as a
   tiebreaker.
2. Use `.map()` with a dictionary to create a new column translating department names to
   single-letter codes.
3. Use `pd.cut` to bin `salary` into `"Low"`, `"Mid"`, `"High"` categories with your own
   boundaries.
4. Use the `.str` accessor to create a column that's `True` if the person's name has more than
   5 characters.

Continue to **`06_groupby_and_aggregation.ipynb`** next.